# Chapter 10 — Robustness

Source: *Effective Python*, 3rd Edition (Brett Slatkin), Chapter 10.

---


### Item 80 — Take Advantage of Each Block in `try`/`except`/`else`/`finally`

All four clauses have distinct jobs: `try` is the risky code; `except` handles a specific failure; `else` runs only if `try` succeeded with **no** exception (keeping "what to do on success" separate from "what might fail"); `finally` always runs, success or failure, for cleanup that must happen no matter what. Using all four deliberately (not just `try`/`except`) keeps each concern in its own place.


In [1]:
def load_and_process(path):
    try:
        f = open(path)
    except FileNotFoundError:
        print('handling: file not found')
        return None
    else:
        # only reached if open() succeeded -- success-path logic, separate from error handling
        try:
            data = f.read()
            print('processed', len(data), 'bytes')
            return data
        finally:
            f.close()   # always runs, whether read() succeeded or raised

from pathlib import Path
Path('robust_demo.txt').write_text('hello world')
load_and_process('robust_demo.txt')
load_and_process('missing_file.txt')
Path('robust_demo.txt').unlink()


processed 11 bytes
handling: file not found


### Item 81 — `assert` Internal Assumptions and `raise` Missed Expectations

`assert condition` is for catching **your own bugs** — invariants that should be impossible to violate if your code is correct (and which can be stripped out entirely when Python runs with `-O`, see Item 90). `raise` is for **expected failure conditions** that come from the outside world (bad user input, a missing file) and must always be checked, in every build, since they're not bugs — they're normal things that can go wrong.


In [2]:
def get_batch(item_count, batch_size):
    # an internal invariant WE control -- if this fails, it's a bug in OUR code
    assert batch_size > 0, 'batch_size must be positive (internal bug if this fires)'
    return (item_count + batch_size - 1) // batch_size

print(get_batch(100, 10))

def withdraw(balance, amount):
    # a condition from the OUTSIDE world (user input) -- always possible, not a bug, must always be checked
    if amount > balance:
        raise ValueError(f'Cannot withdraw {amount}: only {balance} available')
    return balance - amount

try:
    withdraw(50, 100)
except ValueError as e:
    print('Expected external failure, always checked:', e)


10
Expected external failure, always checked: Cannot withdraw 100: only 50 available


### Item 82 — Consider `contextlib` and `with` Statements for Reusable `try`/`finally` Behavior

Repeating the same `try`/`finally` cleanup pattern (acquire a resource, guarantee it's released) across a codebase is exactly what a context manager automates. `contextlib.contextmanager` turns a generator function into a `with`-compatible context manager with minimal boilerplate: code before `yield` is the setup, code after is the guaranteed cleanup — reusable anywhere with a simple `with my_context():`.


In [3]:
from contextlib import contextmanager
import time

@contextmanager
def timed_operation(name):
    start = time.time()
    print(f'starting {name}...')
    try:
        yield        # the code inside the `with` block runs here
    finally:
        elapsed = time.time() - start
        print(f'{name} finished in {elapsed:.3f}s')   # ALWAYS runs, success or exception

with timed_operation('data processing'):
    total = sum(range(1_000_000))
print('result:', total)

try:
    with timed_operation('a failing operation'):
        raise ValueError('oops')
except ValueError:
    print('exception propagated, but cleanup (the timing print) still ran first')


starting data processing...
data processing finished in 0.012s
result: 499999500000
starting a failing operation...
a failing operation finished in 0.000s
exception propagated, but cleanup (the timing print) still ran first


### Item 83 — Always Make `try` Blocks as Short as Possible

The more lines inside a `try` block, the harder it is to know which line actually raised the exception you're catching — a long `try` might accidentally swallow an unrelated bug from a totally different line as if it were the expected failure. Keep only the single call that can actually raise the specific exception you're handling inside `try`; move everything else outside it.


In [4]:
def parse_config_bad(raw):
    try:
        # multiple things that could go wrong are bundled together -- which one actually failed?
        stripped = raw.strip()
        parts = stripped.split('=')
        key = parts[0].upper()          # could raise IndexError if there's no '='
        value = int(parts[1])            # could raise ValueError OR IndexError
        return {key: value}
    except (IndexError, ValueError):
        return None   # you can't tell from here which specific step failed

def parse_config_good(raw):
    stripped = raw.strip()
    parts = stripped.split('=')
    if len(parts) != 2:
        return None   # handled explicitly, outside any try -- not disguised as an exception

    key = parts[0].upper()
    try:
        value = int(parts[1])   # ONLY the genuinely risky conversion is inside try
    except ValueError:
        return None
    return {key: value}

print(parse_config_bad('PORT=8080'), parse_config_good('PORT=8080'))
print(parse_config_bad('bad_input'), parse_config_good('bad_input'))


{'PORT': 8080} {'PORT': 8080}
None None


### Item 84 — Beware of Exception Variables Disappearing

`except Exception as e:` behaves unusually: Python automatically **deletes** `e` at the end of the `except` block (this is unlike every other variable assignment in Python) — so if you try to reference `e` again after the block, or store a reference to it for later, you'll get a `NameError` (or, if you stored it in a way that seems to work, subtle bugs). If you need the exception afterward, explicitly assign it to a different variable inside the block.


In [5]:
try:
    raise ValueError('something broke')
except ValueError as e:
    print('inside the except block:', e)

try:
    print(e)   # NameError! `e` was automatically deleted when the except block ended
except NameError as err:
    print(f'{type(err).__name__} after the block:', err)

# Fix: explicitly save it to a DIFFERENT name if you need it after the except block
saved_error = None
try:
    raise ValueError('something broke again')
except ValueError as e:
    saved_error = e   # copy the reference to a name that WON'T be auto-deleted

print('still accessible:', saved_error)


inside the except block: something broke
NameError after the block: name 'e' is not defined
still accessible: something broke again


### Item 85 — Beware of Catching the `Exception` Class

`except Exception:` catches almost everything (every ordinary error in your program) — which makes it tempting as a catch-all, but it also silently swallows bugs you never anticipated (typos, wrong types, logic errors), hiding them instead of surfacing them. Catch the most specific exception type(s) you actually expect and know how to handle; let genuinely unexpected exceptions propagate so they're visible.


In [6]:
def divide_overly_broad(a, b):
    try:
        return a / b
    except Exception:                # catches ZeroDivisionError, but ALSO TypeError, and everything else
        return None                   # a real bug (e.g. passing a string) is silently hidden as "None"

print(divide_overly_broad(10, 0))       # intended case: division by zero
print(divide_overly_broad(10, 'oops'))  # BUG: wrong type entirely, silently hidden as if it were the same case

def divide_specific(a, b):
    try:
        return a / b
    except ZeroDivisionError:          # only catches what we actually expect and know how to handle
        return None

print(divide_specific(10, 0))
try:
    divide_specific(10, 'oops')         # the real bug is now VISIBLE, not silently hidden
except TypeError as e:
    print('Bug surfaced instead of hidden:', e)


None
None
None
Bug surfaced instead of hidden: unsupported operand type(s) for /: 'int' and 'str'


### Item 86 — Understand the Difference Between `Exception` and `BaseException`

`Exception` is the base of almost every error your own code should raise or catch. `BaseException` is one level higher, and also covers `SystemExit` and `KeyboardInterrupt` — things that mean "the program is being deliberately stopped," not "an error occurred." Catching `BaseException` (or a bare `except:`) accidentally intercepts `Ctrl+C`/`sys.exit()` too, preventing a program from actually being able to quit when asked. Always catch `Exception` (or more specific types), never `BaseException`, unless you have a very deliberate reason.


In [7]:
print(issubclass(Exception, BaseException))      # Exception IS a BaseException...
print(issubclass(KeyboardInterrupt, Exception))   # ...but KeyboardInterrupt is NOT an Exception

def risky_catch_exception():
    try:
        raise KeyboardInterrupt   # simulates a user pressing Ctrl+C
    except Exception:              # does NOT catch KeyboardInterrupt -- as intended
        print('caught by Exception (this should NOT print)')

try:
    risky_catch_exception()
except KeyboardInterrupt:
    print('KeyboardInterrupt correctly propagated past `except Exception` -- program CAN still be stopped')


True
False
KeyboardInterrupt correctly propagated past `except Exception` -- program CAN still be stopped


### Item 87 — Use `traceback` for Enhanced Exception Reporting

The standard `traceback` module lets you capture and format a full stack trace programmatically (not just let Python print it to stderr automatically) — useful for logging exceptions with full context (which function, which line) into your own log files/monitoring systems, rather than just printing a bare error message and losing where it actually happened.


In [8]:
import traceback

def inner():
    raise ValueError('deep failure')

def outer():
    inner()

try:
    outer()
except ValueError:
    formatted = traceback.format_exc()   # the FULL trace as a string, ready to log anywhere
    print('Captured full traceback for logging:')
    print(formatted)


Captured full traceback for logging:
Traceback (most recent call last):
  File "/sessions/festive-peaceful-johnson/tmp/ipykernel_11/2664763575.py", line 10, in <module>
    outer()
  File "/sessions/festive-peaceful-johnson/tmp/ipykernel_11/2664763575.py", line 7, in outer
    inner()
  File "/sessions/festive-peaceful-johnson/tmp/ipykernel_11/2664763575.py", line 4, in inner
    raise ValueError('deep failure')
ValueError: deep failure



### Item 88 — Consider Explicitly Chaining Exceptions to Clarify Tracebacks

When handling one exception triggers raising a *different* one (e.g., catching a low-level `KeyError` and raising a higher-level `ConfigError`), use `raise NewError(...) from original_error` — Python then shows both exceptions in the traceback with a clear "the above exception was the direct cause" note, instead of only showing the new exception and losing the original root cause entirely.


In [9]:
class ConfigError(Exception):
    pass

def load_setting(config, key):
    try:
        return config[key]
    except KeyError as original:
        # `from original` preserves the ROOT CAUSE in the traceback, not just the new exception
        raise ConfigError(f'Missing required setting: {key}') from original

try:
    load_setting({'host': 'localhost'}, 'port')
except ConfigError as e:
    print('New exception:', e)
    print('Root cause preserved via __cause__:', e.__cause__)


New exception: Missing required setting: port
Root cause preserved via __cause__: 'port'


### Item 89 — Always Pass Resources into Generators and Have Callers Clean Them Up Outside

A generator that opens its own resource internally (e.g., opens a file itself) puts the caller in an awkward position — if they stop iterating early (`break`), the resource might never get cleaned up, since the generator's cleanup code (in a `finally`) only runs once the generator is fully exhausted or explicitly closed. Instead, have the **caller** open the resource (typically via `with`) and simply pass it into the generator — the caller then unambiguously controls its lifetime, regardless of how the generator is consumed.


In [10]:
from pathlib import Path

Path('gen_resource_demo.txt').write_text('line1\nline2\nline3\n')

# Preferred: the CALLER owns the resource via `with`; the generator just receives an already-open file
def read_lines(open_file):
    for line in open_file:
        yield line.strip()

with open('gen_resource_demo.txt') as f:    # caller controls the resource's lifetime explicitly
    for i, line in enumerate(read_lines(f)):
        print(line)
        if i == 0:
            break   # even stopping early, the `with` block still guarantees the file gets closed

print('file closed after the with block:', f.closed)
Path('gen_resource_demo.txt').unlink()


line1
file closed after the with block: True


### Item 90 — Never Set `__debug__` to `False`

`__debug__` is a built-in constant, `True` by default, that `assert` statements are compiled against — running Python with the `-O` flag sets it to `False` and strips every `assert` from the bytecode entirely (a performance optimization). Manually trying to set `__debug__ = False` yourself is actually a `SyntaxError` (Python protects this), and even if it were possible, doing so removes all your safety checks silently — the book's point is: don't run production code with `-O` relying on asserts for correctness, since they may not even execute.


In [11]:
print('__debug__ is:', __debug__)   # True in normal execution

try:
    exec('__debug__ = False')   # Python actively disallows reassigning this
except SyntaxError as e:
    print('SyntaxError (as expected, Python protects this):', e)

def validate(x):
    assert x > 0, 'x must be positive'   # this check literally DISAPPEARS if run with `python -O`
    return x * 2

print(validate(5))
print('Running with `python -O script.py` would silently skip the assert above -- never rely on')
print('asserts for checks that MUST always happen; use explicit `if ... raise ...` for those instead.')


__debug__ is: True
SyntaxError (as expected, Python protects this): cannot assign to __debug__ (<string>, line 1)
10
Running with `python -O script.py` would silently skip the assert above -- never rely on
asserts for checks that MUST always happen; use explicit `if ... raise ...` for those instead.


### Item 91 — Avoid `exec` and `eval` Unless You're Building a Developer Tool

`exec()`/`eval()` run arbitrary Python source from a string at runtime — extremely powerful, and extremely dangerous: executing anything derived from untrusted input (user text, network data) is a direct code-injection vulnerability, and even trusted dynamic code is hard to debug, statically analyze, or optimize. The book's guidance: reserve these for genuine developer-tooling use cases (a REPL, a plugin system with fully trusted code), never for regular application logic.


In [12]:
# eval(): evaluates a single expression and returns its value
result = eval('2 + 3 * 4')
print(result)

# exec(): runs a full block of statements
namespace = {}
exec('x = 10\ny = x * 2', namespace)
print(namespace['x'], namespace['y'])

# The danger: NEVER do this with untrusted input -- it's arbitrary code execution
untrusted_input = "__import__('os').getcwd()"   # imagine this came from a user-submitted form
print('what eval() would do with untrusted input:', eval(untrusted_input))
print('This is exactly why eval/exec on untrusted input is a critical security vulnerability.')


14
10 20
what eval() would do with untrusted input: /tmp
This is exactly why eval/exec on untrusted input is a critical security vulnerability.
